## The Responses API by OpenAI

In [ ]:
from openai import OpenAI
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv  # Import the specific functions from dotenv
import time

In [ ]:
# find_dotenv() will automatically climb up from 'developer_path' 
# to the root folder to find your .env file flawlessly.
load_dotenv(find_dotenv())

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")

# Safety check to make sure it loaded
if not api_key:
    raise ValueError("API Key is still missing! Double-check the variable name inside your .env file.")

# Initialize the client
client = OpenAI(api_key=api_key)
print("Connected successfully! OpenAI client is ready.")

Connected successfully! OpenAI client is ready.


In [ ]:
# test time taken
start_time = time.time()

# this is the response API
response = client.responses.create(
    model="gpt-5.1-mini",
    reasoning={"effort": "medium"}, # param none, low, medium, high, or max
    max_output_tokens=200,
    instructions="Be concise and provide a brief summary of the following text.",
    input="The OpenAI Responses API is a powerful tool that allows developers to interact with OpenAI's language models. It provides a simple interface for sending prompts and receiving generated text, making it easier to integrate AI capabilities into applications. The API supports various models, each with different strengths and capabilities, enabling users to choose the best fit for their specific use case.",
    previous_response_id=None # getting from a different response variable, example: response1
)

print(response.output_text) # using the output_text attribute to get the generated text from the response object

# metadata
print(response.usage.output_tokens)  # Number of tokens in the output
print(response.id)  # Unique identifier for the response

# output form
print(response.output)  # Full output object, which may include additional details beyond just the text

# time taken validation
runtime = time.time() - start_time
print(f"\nRuntime: {runtime:.2f} seconds")

### Mastering Response Parameters

- No reasoning LLMs: generate tokens with the highest probability
- With reasoning LLMs: breaks down the problem into steps
- Model <-> Reasoning <-> Tokens = Total Cost

### Conversation Chaining

- Mechanism: Control & Retrieval method 

In [ ]:
system_prompt = "You are a helpful assistant that summarizes text."
latest_response_id = None

while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "exit":
        break

    response = client.responses.create(
        model="gpt-5.1-mini",
        reasoning={"effort": "medium"},
        max_output_tokens=200,
        instructions=system_prompt,
        input=user_input,
        previous_response_id=latest_response_id
    )

    print(f"\nAI: {response.output_text}")
    latest_response_id = response.id

### Response Items: The Building Blocks
- Reasoning
- Message
- Status

In [ ]:
# checking the content inside the param 
for item in response.output:
    if item.type == "message": # reasoning, tool, or message
        if item.status == "incomplete":
            print("Message cut-off!")

for item in response.output:
    if item.type == "reasoning": 
        if item.summary:
            print(f"Reasoning Summary: {item.summary[0]}")
        else:
            print("No reasoning summary available.")    
    if item.type == "message":
            print(f"Assistant: {item.content[0].output_text}")

### Web Search 
- Need for Up-to-Date Information

In [ ]:
response = client.responses.create(
    model="gpt-5.1-mini",
    reasoning={"effort": "medium"},
    max_output_tokens=200,
    tool=[{"type":"web_search"}],
    input="What is the current temperature in Cyberjaya, Malaysia?",
    include=["web_search_call.action.sources"]
)

# to get the sources from the web search tool
for item in response.output:
    if item.type == "web_search_call":
        print(item)

### Function-Calling Tools
- Function-Calling Tools: LLMs calling programmed functions
- Steps/Workflow:
    1. Tools Defintions & User Messages
    2. Tool Call & Arguments
    3. Execute Functions
    4. Send Functions Results
    5. Final Response

![alt text](<Screenshot 2026-08-17 163633.png>)

In [ ]:
# example convert_currency(), params: amount, from_currency, to_currency
# example time zone https://api.opentimezone.com/convert
import requests

def convert_currency(amount, from_currency, to_currency) -> str:
    url = f"https://api.frankfurter.dev/v1/latest?base={from_currency}&symbols={to_currency}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        rate = data["rates"].get(to_currency)

        if rate is None:
            return f"Could not find conversion rate for {to_currency}."

        converted_amount = amount * rate
        return f"{amount} {from_currency} is approximately {converted_amount:.2f} {to_currency}."

    except requests.exceptions.RequestException as e:
        print("Error occurred while fetching currency conversion data.")
        return None

In [ ]:
# test output
conversion_result = convert_currency(19, "USD", "MYR")
print(conversion_result)

19 USD is approximately 77.63 MYR.


### Function-Calling Tools: Full Workflow

In [ ]:
# tools: LLM manual

tools = [
    {
        "type":"function",
        "name":"convert_currency",
        "description":"Converts an amount from one currency to another using the Frankfurter API.",
        "parameters":{
            "type":"object",
            "properties":{
                "amount":{
                    "type":"number",
                    "description":"The amount to convert."
                },
                "from_currency":{
                    "type":"string",
                    "description":"The currency code to convert from."
                },
                "to_currency":{
                    "type":"string",
                    "description":"The currency code to convert to."
                }
            },
            "required":["amount", "from_currency", "to_currency"],
            "additionalProperties": False
        }
    }
]

In [ ]:
# structured role based
messages = [
    {
        "role": "user",
        "content": "Convert 100 USD to MYR."
    }
]

response = client.responses.create(
    model="gpt-5.1-mini",
    reasoning={"effort": "medium"},
    max_output_tokens=200,
    tool=tools,
    input="Convert 100 USD to MYR.", # single input or messages=messages for multiple inputs
    include=["tool_call.action"]
)


In [ ]:
for item in response.output:
    if item.type == "function_call":
        if item.name == "convert_currency":
            currency_result = convert_currency(**json.loads(item.arguments))

            messages.append({
                "type":"function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                    "convert_currency": currency_result})
            })

response = client.responses.create(
    model="gpt-5.1-mini",
    instructions="You are a helpful assistant that converts currencies. Please provide the conversion result.",
    tools=tools,
    input=messages,
)

print(response.output_text)  # This should contain the final conversion result from the assistant

### Generating Structured Outputs (JSON)
- Step 1: Defining the Output Schema

In [ ]:
from pydantic import BaseModel, Field

class QuizResult(BaseModel):
    score: int = Field(..., description="The score obtained in the quiz.")
    passed: bool = Field(..., description="Whether the quiz was passed.")
    feedback: str = Field(..., description="Feedback about the quiz performance.")

class Mistake(BaseModel):
    word: str = Field(..., description="The word that was misspelled.")
    student_answer: str = Field(..., description="The student's answer for the misspelled word.")
    correct_answer: str = Field(..., description="The correct spelling of the word.")

class DetailedQuizResult(BaseModel):
    score: int = Field(..., description="The score obtained in the quiz.")
    passed: bool = Field(..., description="Whether the quiz was passed.")
    feedback: str = Field(..., description="Feedback about the quiz performance.")
    mistakes: list[Mistake] = Field(..., description="List of mistakes made in the quiz.")

- Step 2: Sending the Request

In [ ]:
response = client.responses.parse(
    model="gpt-5.4.mini",
    instructions="You are a helpful assistant that provides quiz results. Please provide the quiz result in the specified format.",
    input="""
    1. casa = house
    2. perro = dog
    3. gato = car
    4. libro = book
    5. aqua = water
    """,
    text_format=DetailedQuizResult
)

- Step 3: Extracting Results

In [ ]:
result = response.output_parsed
print(f"Score: {result.score}, Passed: {result.passed}, Feedback: {result.feedback}")

### Streaming with Semantics Events

![alt text](<Screenshot 2026-08-19 124529.png>)

Basic Text Streaming

In [ ]:
prompt = "Explain the season year concisely to a 10-year-old in 2 sentences."

with client.responses.create(
    model="gpt-5.1-mini",
    reasoning={"effort": "medium"},
    max_output_tokens=200,
    instructions="Be concise and provide a brief summary of the following text.",
    input=prompt,
    stream=True
) as stream: 
    for event in stream:
        # single event type
        # example event type: delta
        if event.type == "response.output_text.delta":
            current_text += event.delta
            print(current_text)

In [ ]:
# example: handling multiple events
prompt = "Explain the season year concisely to a 10-year-old in 2 sentences."

with client.responses.create(
    model="gpt-5.1-mini",
    reasoning={"effort": "medium"},
    max_output_tokens=200,
    instructions="Be concise and provide a brief summary of the following text.",
    input=prompt,
    stream=True,
    tools=tools # add param tools to enable tool usage in the streaming response
) as stream: 
    for event in stream:
        # multiple event types
        # example event type: created
        if event.type == "response.created":
            print("Response started...\n")

        # example event type: done
        elif event.type == "response.output_text.done":
            print("\nText block completed.")

        # example event type: delta
        elif event.type == "response.function_call_arguments.delta":
            current_args += event.delta
            print(f"Streaming args: {current_args}")

        # example event type: function call
        elif event.type == "response.function_call_arguments.done":
            print(f"Final arguments: {event.arguments}")

        # example event type; completed
        elif event.type == "response.completed":
            print("\nResponse completed.")



### Input Flexibility & Multimodality

Input Flexibility

In [ ]:
# prompt as Strings
response = client.responses.create(
    model="gpt-5.4-mini",
    input="How old is the earth?",
    instructions="You are a helpful assistant that answers questions about the earth. Please provide the answer.",
    max_output_tokens=100
)

# role-based prompt
response = client.responses.create(
    model="gpt-5.4-mini",
    input=[ # adding guard rails
        {"role": "system", "content": "You are a helpful assistant that answers questions about the earth."},
        {"role": "user", "content": "How old is the earth?"}
    ]
)

# role-based message
response = client.responses.create(
    model="gpt-5.4-mini",
    input=[ # adding guard rails
        {"role": "system", "content": "You are a helpful assistant that answers questions about the earth."},
        {"role": "user", "content": "How old is the earth?"},
        {"role": "assistant", "content": "The earth is approximately 4.54 billion years old."}, # one-shot prompt
        {"role": "user", "content": "How old is the sun?"}
    ]
)


In [ ]:
system_prompt = "You are a helpful assistant that answers questions about the earth."
latest_response_id = None

while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "exit":
        break

    response = client.responses.create(
        model="gpt-5.4-mini",
        reasoning={"effort": "medium"},
        max_output_tokens=200,
        instructions=system_prompt,
        input=user_input,
        previous_response_id=latest_response_id
    )

    print(f"\nAI: {response.output_text}")
    latest_response_id = response.id

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions about the earth."},
    {"role": "user", "content": "How old is the earth?"},
]

while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "exit":
        break

    messages.append({"role": "user", "content": user_input})

    response = client.responses.create(
        model="gpt-5.4-mini",
        reasoning={"effort": "medium"},
        max_output_tokens=200,
        instructions=system_prompt,
        input=messages,
    )

    print(f"\nAI: {response.output_text}")
    messages.append({"role": "assistant", "content": response.output_text})